# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by its Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print summary
md = dataset.metadata
print(f"Dataset: {md.name}\n")
print(f"Identifier: {md.identifier}")
print(f"Version: {md.version}")
print(f"Published: {md.datePublished}")
print(f"Spatial coverage: {md.spatialCoverage}")
print(f"Description: {md.description}\n")
print("Cite as:")
print(md.citeAs)

## 2. Data Overview

List all available record sets and their fields, showing their `@id` references.

> **Note:** All references below use the Croissant schema `@id` as required.

In [ ]:
# Find and list all record sets, their @ids, and fields
record_sets = []
if hasattr(md, "recordSet") and md.recordSet:
    for rs in md.recordSet:
        # rs is a RecordSet object
        print(f"RecordSet: {rs.name}  (@id: {rs['@id']})")
        print(f"  Description: {getattr(rs, 'description', 'N/A')}")
        print(f"  Fields:")
        if hasattr(rs, "field") and rs.field:
            for f in rs.field:
                print(f"    - {f.name} (@id: {f['@id']}) [type: {f.dataType}]" )
        print()
        record_sets.append(rs)
else:
    print("No record sets defined in this Croissant metadata.")

# Capture the list of record set @ids for later use
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction

Load data from each record set into pandas DataFrames. Refer to each record set by its `@id`.

In [ ]:
# If no record sets were found above, try to reload them from the dataset object
if not record_set_ids:
    # Try to access via dataset.metadata.recordSet
    record_sets = getattr(dataset.metadata, "recordSet", [])
    record_set_ids = [rs['@id'] for rs in record_sets if '@id' in rs]

dataframes = dict()
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Loading data for RecordSet '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        print(f"Columns: {df.columns.tolist()}")
        dataframes[rs_id] = df
        display(df.head())
    else:
        print(f"No records found for record set '@id': {rs_id}\n")

if not dataframes:
    print("No tabular data available in this dataset. The recordSet definitions may be absent from the schema.")

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing: filter, normalize, and group records from a chosen record set by their `@id`. 

If there are no tabular record sets, a message will be shown.

In [ ]:
if dataframes:
    # Use the first available record set for demonstration
    chosen_rs_id = list(dataframes.keys())[0]
    df = dataframes[chosen_rs_id]
    print(f"Using record set '@id': {chosen_rs_id}\n")

    # Find a numeric field using Croissant metadata
    rs_meta = next((rs for rs in record_sets if rs['@id'] == chosen_rs_id), None)
    numeric_field_id = None
    group_field_id = None
    if rs_meta and hasattr(rs_meta, 'field') and rs_meta.field:
        for f in rs_meta.field:
            # Use 'Float' or 'Integer'
            if getattr(f, 'dataType', None) in ('schema:Float', 'schema:Integer'):
                numeric_field_id = f['@id']
                if f['@id'] in df.columns:
                    break
    if not numeric_field_id:
        # Default to first column if none found
        numeric_cols = df.select_dtypes(include='number').columns
        if len(numeric_cols) > 0:
            numeric_field_id = numeric_cols[0]
        else:
            print("No numeric field available for analysis.")
            numeric_field_id = None

    # Optionally, find a group field
    if rs_meta and hasattr(rs_meta, 'field') and rs_meta.field:
        for f in rs_meta.field:
            if getattr(f, 'dataType', None) == 'schema:Text' and f['@id'] in df.columns:
                group_field_id = f['@id']
                break

    if numeric_field_id and numeric_field_id in df.columns:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First 5 normalized values of {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No dataframes available to analyze.")

## 5. Visualization

Plot distributions or relationships between fields from a selected record set, using `@id` fields for reference.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization (if data exists and a numeric field was found)
if dataframes and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No suitable data available for visualization.")

## 6. Conclusion

In this notebook, we've shown how to use the `mlcroissant` library to load metadata, overview the dataset structure by `@id`, extract records, and perform basic EDA and visualization tasks using standardized Croissant references.

This dataset covers ordered logistic regression outputs for rangeland management adoption predictors in Northern Kenya. For further model analysis or integration, continue exploring the data fields and distributions described in this notebook.